<a href="https://colab.research.google.com/github/CSB-KaracaLab/af-distogram-flexibility/blob/main/AF_distogram_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold distograms and hinge motions, in Colab

**Companion notebook to [`CSB-KaracaLab/af-distogram-flexibility`](https://github.com/CSB-KaracaLab/af-distogram-flexibility).**

AlphaFold's distogram head predicts, for every residue pair, a distribution over distances
rather than one number. That distribution can be broad where the final coordinates look
perfectly confident — and in a protein that hinges on binding, the breadth sits exactly
across the hinge. This notebook walks that idea end to end on one of the paper's own
systems, and leaves you set up to do the same on yours.

> Savaş B, Barlas AB, Karaca E. *Exploring the Potential of AlphaFold Distograms for
> Predicting Binding-induced Hinge Motions.* **FEBS Letters** ·
> [doi:10.1002/1873-3468.70297](https://doi.org/10.1002/1873-3468.70297)

### The worked example

**MIA40**, measured across the CB atoms of residues **7 and 10**, the pair the paper uses
for this system. The notebook:

1. downloads the MIA40 structure ensembles the paper deposited on
   [Zenodo](https://doi.org/10.5281/zenodo.17638774) — 3,561 models from nine different
   sampling methods;
2. measures the 7–10 distance in every one of them with the repository's own script, and
   **checks the result against the table the paper deposited**;
3. puts that measured spread next to the AlphaFold distogram for the same pair.

Step 2 is a real reproduction, not a demonstration: if the numbers stop matching, the
notebook says so.

### What you need

A Colab CPU runtime. No GPU — nothing here runs AlphaFold; it reads what AlphaFold
produced. The download is about 82 MB and the measurement takes a few minutes.

In [ ]:
#@title 1. Set up { display-mode: "form" }
#@markdown Clones the repository and installs PyMOL (for the distance measurements),
#@markdown numpy, pandas and matplotlib. About two minutes.
import os, subprocess, sys, time

t0 = time.time()
if not os.path.isdir("af-distogram-flexibility"):
    subprocess.run(["git", "clone", "--quiet", "--depth", "1",
                    "https://github.com/CSB-KaracaLab/af-distogram-flexibility.git"],
                   check=True)
REPO = os.path.abspath("af-distogram-flexibility")
SCRIPTS = os.path.join(REPO, "scripts")

if not os.path.isfile("DEPS_READY"):
    # pymol-open-source ships manylinux wheels, so it installs with pip here even though
    # it normally wants conda
    os.system("pip install -q pymol-open-source 2>&1 | tail -2")
    os.system(f"pip install -q -r {REPO}/requirements.txt matplotlib")
    os.system("touch DEPS_READY")

import pymol  # noqa: E402
print(f"repository : {REPO}")
print(f"PyMOL      : {pymol.cmd.get_version()[0]}")
print(f"ready in {time.time() - t0:.0f} s")

In [ ]:
#@title 2. Fetch the deposited MIA40 ensembles { display-mode: "form" }
#@markdown 82 MB from Zenodo: 3,561 predicted structures of MIA40 from nine sampling
#@markdown methods — AF2 and AF3 enhanced sampling, and the generative models the paper
#@markdown compares against.
import collections, os, urllib.request, zipfile
from pathlib import Path

ZENODO = "https://zenodo.org/records/17638774/files/MIA40.zip"
archive = Path("MIA40.zip")

if not archive.exists():
    print("downloading ...", flush=True)
    urllib.request.urlretrieve(ZENODO, archive)
print(f"{archive} : {archive.stat().st_size / 1e6:.0f} MB")

if not Path("MIA40").is_dir():
    with zipfile.ZipFile(archive) as z:
        z.extractall(".")

ENSEMBLES = Path("MIA40")
counts = collections.Counter(p.parent.name for p in ENSEMBLES.rglob("*.pdb"))
print(f"\n{sum(counts.values())} structures in {len(counts)} ensembles:")
for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<14} {n:>5}")

In [ ]:
#@title 3. Measure residues 7 and 10 in all of them { display-mode: "form" }
#@markdown This is the repository's own `dist-calc-two-res.py`, pointed at the directory.
#@markdown PyMOL is started once and every structure measured in turn, so the whole
#@markdown ensemble takes a few minutes rather than an hour.
residue1 = 7  #@param {type:"integer"}
residue2 = 10  #@param {type:"integer"}
atom = "CB"  #@param ["CB", "CA"]
#@markdown Glycine has no CB. If either residue is a glycine, measure on CA instead.
import subprocess, time
from pathlib import Path

out_csv = Path(f"MIA40_res{residue1}_{residue2}_recomputed.csv")
if out_csv.exists():
    out_csv.unlink()                      # the script appends, so start clean

t0 = time.time()
run = subprocess.run(
    [sys.executable, f"{SCRIPTS}/dist-calc-two-res.py",
     "--input_pdb", str(ENSEMBLES),
     "--residue1", str(residue1), "--residue2", str(residue2),
     "--atom", atom, "--output_csv", str(out_csv)],
    capture_output=True, text=True)
print(run.stdout.strip() or run.stderr[-1500:])
print(f"\n{time.time() - t0:.0f} s")

In [ ]:
#@title 4. Does it reproduce the deposited table? { display-mode: "form" }
#@markdown The paper deposited its own measurements for this pair. Recomputing them from
#@markdown the deposited structures should give the same numbers back; this cell says
#@markdown whether it does, rather than assuming it.
import numpy as np, pandas as pd
from pathlib import Path

mine = pd.read_csv(out_csv)
deposited_path = Path(REPO) / "data/pdb-distances/MIA40_res7_10_CB_measurements.csv"

if (residue1, residue2, atom) != (7, 10, "CB") or not deposited_path.is_file():
    print(f"No deposited table for {atom}{residue1}-{atom}{residue2}; "
          "the comparison below only applies to the published CB7-CB10 pair.")
    measured = mine
else:
    deposited = pd.read_csv(deposited_path).rename(columns={"Distance": "deposited"})
    merged = mine.merge(deposited[["PDB file", "deposited"]], on="PDB file", how="outer",
                        indicator=True)
    both = merged[merged._merge == "both"].copy()
    both["difference"] = (both.Distance - both.deposited).abs()

    print(f"recomputed here     : {len(mine)}")
    print(f"deposited with paper: {len(deposited)}")
    print(f"matched by filename : {len(both)}")
    print(f"only here           : {(merged._merge == 'left_only').sum()}")
    print(f"only deposited      : {(merged._merge == 'right_only').sum()}")
    print()
    agree = (both.difference < 0.01).sum()
    print(f"agree within 0.01 A : {agree} / {len(both)}")
    print(f"largest difference  : {both.difference.max():.4f} A")
    if agree == len(both):
        print("\nEvery deposited measurement reproduces from the deposited structures.")
    else:
        worst = both.nlargest(5, "difference")[["PDB file", "Distance", "deposited",
                                                "difference"]]
        print("\nLargest disagreements:")
        display(worst)
    measured = mine

In [ ]:
#@title 5. How far apart do the methods put them? { display-mode: "form" }
#@markdown One dot per structure, by the method that produced it. This is the sampled
#@markdown spread: what each strategy actually explored for this residue pair.
import matplotlib.pyplot as plt
import numpy as np

INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
HUE = "#2a78d6"            # one hue: the methods are positions on the axis, not a palette

order = (measured.groupby("Folder").Distance.median().sort_values().index.tolist())
fig, ax = plt.subplots(figsize=(7.6, 0.46 * len(order) + 1.6), dpi=140)
rng = np.random.default_rng(0)

for y, method in enumerate(order):
    values = measured.loc[measured.Folder == method, "Distance"].to_numpy()
    ax.scatter(values, y + rng.uniform(-0.17, 0.17, len(values)), s=9, alpha=0.35,
               color=HUE, linewidths=0, zorder=3)
    ax.plot([np.median(values)] * 2, [y - 0.3, y + 0.3], color=INK, lw=2,
            solid_capstyle="round", zorder=4)

# the counts go on once everything is drawn, so they share one x and do not drift as
# the axes grow
label_x = ax.get_xlim()[1]
for y, method in enumerate(order):
    n = int((measured.Folder == method).sum())
    ax.text(label_x, y, f"  n={n}", va="center", fontsize=8, color=INK_SOFT)

ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlabel(f"{atom}{residue1}–{atom}{residue2} distance (Å)", fontsize=9.5, color=INK_SOFT)
ax.set_title(f"MIA40: {len(measured)} structures, the black bar is each method's median",
             fontsize=10.5, color=INK, pad=12, loc="left")
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=INK_SOFT, length=0)
plt.tight_layout()
plt.show()

summary = (measured.groupby("Folder").Distance
           .agg(n="count", median="median", spread=lambda s: s.max() - s.min())
           .reindex(order).round(2))
display(summary)

In [ ]:
#@title 6. The distogram beside the sampled spread { display-mode: "form" }
#@markdown The deposited distograms for this pair, averaged over the runs of each method,
#@markdown drawn against the distances actually sampled above. The paper's argument is
#@markdown that the distogram is already broad across a hinge — before any of the
#@markdown enhanced sampling is done.
import pandas as pd

dgram_path = Path(REPO) / "data/distograms/MIA40_distograms_res7_10.csv"
dgram = pd.read_csv(dgram_path)
axis = dgram["Distance_Angstrom"].to_numpy()
runs = dgram.drop(columns=["Distance_Angstrom"])
print(f"{runs.shape[1]} predictions, {runs.shape[0]} distance bins")

def method_of(column):
    return column.split("-")[0].split("_")[0]

families = sorted({method_of(c) for c in runs.columns})
mean_curve = runs.mean(axis=1).to_numpy()

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7.6, 5.4), dpi=140, sharex=True,
                                  gridspec_kw={"height_ratios": [1.15, 1]})

for family in families:
    cols = [c for c in runs.columns if method_of(c) == family]
    top.plot(axis, runs[cols].mean(axis=1), lw=1.2, alpha=0.5, color=INK_SOFT)
top.plot(axis, mean_curve, lw=2.4, color=HUE, label="mean over all predictions")
top.fill_between(axis, 0, mean_curve, color=HUE, alpha=0.10)
top.set_ylabel("distogram probability", fontsize=9.5, color=INK_SOFT)
top.legend(frameon=False, fontsize=8.5, labelcolor=INK_SOFT, loc="upper right")
top.set_title("MIA40 residues 7–10: what AlphaFold predicted, and what was sampled",
              fontsize=10.5, color=INK, pad=12, loc="left")

bottom.hist(measured.Distance, bins=np.arange(axis.min(), axis.max() + 0.3125, 0.3125),
            color=INK_SOFT, alpha=0.55)
bottom.set_ylabel("structures", fontsize=9.5, color=INK_SOFT)
bottom.set_xlabel("Cβ–Cβ distance (Å)", fontsize=9.5, color=INK_SOFT)

for ax_ in (top, bottom):
    ax_.set_axisbelow(True)
    ax_.yaxis.grid(True, color=GRID, lw=0.8)
    for side in ("top", "right"):
        ax_.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax_.spines[side].set_color(GRID)
    ax_.tick_params(colors=INK_SOFT, length=0)
# the distogram axis runs to 22 A, but nearly all of the signal sits well below that;
# clip to the range that holds the predictions and the sampled distances
upper = max(axis[mean_curve.cumsum() / mean_curve.sum() < 0.999].max(),
            measured.Distance.quantile(0.995)) + 1.0
top.set_xlim(axis.min(), min(upper, axis.max()))
plt.tight_layout()
plt.show()

peak = axis[int(np.argmax(mean_curve))]
# the width of the distogram, as the span holding the central 80% of its probability
cumulative = np.cumsum(mean_curve) / mean_curve.sum()
lo = axis[int(np.searchsorted(cumulative, 0.10))]
hi = axis[int(np.searchsorted(cumulative, 0.90))]
print(f"distogram: most likely {peak:.2f} A, central 80% spans {lo:.2f}-{hi:.2f} A "
      f"({hi - lo:.2f} A wide)")
print(f"sampled  : median {measured.Distance.median():.2f} A, "
      f"10th-90th percentile {measured.Distance.quantile(0.1):.2f}-"
      f"{measured.Distance.quantile(0.9):.2f} A")

In [ ]:
#@title 7. A distogram from your own AlphaFold run { display-mode: "form" }
#@markdown Optional. Upload an AlphaFold output holding distogram logits — ColabFold's
#@markdown `*.pickle`, an AlphaFold `result_model_*.pkl`, or an `.npz` — and pull out any
#@markdown residue pair. Leave it unticked to skip.
upload_a_prediction = False  #@param {type:"boolean"}
pair_a = 7  #@param {type:"integer"}
pair_b = 10  #@param {type:"integer"}

if upload_a_prediction:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    run = subprocess.run([sys.executable, f"{SCRIPTS}/extract_distogram_pair.py",
                          "--input_pickle", name, str(pair_a), str(pair_b)],
                         capture_output=True, text=True)
    print(run.stdout.strip() or run.stderr.strip())
    csv_out = Path(f"{Path(name).stem}_res{pair_a}_{pair_b}.csv")
    if csv_out.is_file():
        own = pd.read_csv(csv_out)
        fig, ax = plt.subplots(figsize=(7, 2.9), dpi=140)
        ax.plot(own.Distance_Angstrom, own.Probability, lw=2, color=HUE)
        ax.fill_between(own.Distance_Angstrom, 0, own.Probability, color=HUE, alpha=0.12)
        ax.set_xlabel("distance (Å)", fontsize=9.5, color=INK_SOFT)
        ax.set_ylabel("probability", fontsize=9.5, color=INK_SOFT)
        ax.set_title(f"{name}: residues {pair_a}–{pair_b}", fontsize=10.5, loc="left",
                     color=INK, pad=10)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(GRID)
        ax.tick_params(colors=INK_SOFT, length=0)
        ax.yaxis.grid(True, color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        plt.tight_layout()
        plt.show()
else:
    print("Skipped. Tick the box above to upload one of your own predictions.")

In [ ]:
#@title 8. Perturb an MSA by masking columns { display-mode: "form" }
#@markdown The third script replays a column-masking perturbation onto an alignment. In
#@markdown the paper the columns came from an AFsample2 run, which records them in its
#@markdown output pickle; `--columns` lets you give them directly, which is what this
#@markdown cell does since no pickle is deposited with the repository.
columns_to_mask = "5,6,7,20,21,22"  #@param {type:"string"}

msa_in = Path(REPO) / "data/msas/mia40_0.a3m"
run = subprocess.run([sys.executable, f"{SCRIPTS}/column_masking.py",
                      "--msa_file", str(msa_in),
                      "--columns", columns_to_mask,
                      "--output", "mia40_perturbed.a3m"],
                     capture_output=True, text=True)
print(run.stdout.strip() or run.stderr.strip())

before = msa_in.read_text().splitlines()
after = Path("mia40_perturbed.a3m").read_text().splitlines()
print("\nthe first few rows, before and after:")
shown = 0
for b, a in zip(before, after):
    if b.startswith((">", "#")):
        continue
    print(f"  before  {b[:72]}")
    print(f"  after   {a[:72]}")
    shown += 1
    if shown == 3:
        break
print("\nThe query (the first sequence) is deliberately left alone, and lowercase letters "
      "are insertions that occupy no alignment column, so they are stepped over when the "
      "columns are located in each row.")

## Using this on your own system

Everything above is driven by three arguments: a structure or a directory of them, and a
residue pair. To run it on your own protein:

1. **Measure the pair across your ensemble.** Point step 3 at a directory of PDB files —
   AlphaFold models, frames from a trajectory, anything PyMOL can read. One row per model
   comes out, so the result is a distribution. If either residue is a glycine, set `atom`
   to `CA`: glycine has no Cβ.
2. **Pull the distogram out of the prediction.** Step 7 takes a ColabFold `.pickle`, an
   AlphaFold `result_model_*.pkl`, or an `.npz`. For a **complex**, remember that the
   distogram runs over the chains concatenated — a residue number from the second chain is
   not its index; add the lengths of the chains before it.
3. **Compare the two.** A distogram that is broad where the sampled structures are also
   broad is the signal the paper is after.

### What is not here

No AlphaFold run. This notebook reads predictions; it does not make them. The enhanced
sampling the paper compares — MinnieFold, AF_cluster, AFsample2, AF3sample2 — each have
their own repositories, and [MinnieFold](https://github.com/CSB-KaracaLab/MinnieFold) is
the lab's own.

### Citing

> Savaş B, Barlas AB, Karaca E. *Exploring the Potential of AlphaFold Distograms for
> Predicting Binding-induced Hinge Motions.* **FEBS Letters** ·
> [doi:10.1002/1873-3468.70297](https://doi.org/10.1002/1873-3468.70297)

The structure ensembles are deposited separately:
[10.5281/zenodo.17638774](https://doi.org/10.5281/zenodo.17638774).